# Orbit Template And Tidal Stripping Demo

This notebook is a lightweight orbit workflow built from reusable package code.
It mirrors the old `orbit.ipynb` goal of visualizing an orbit and shape evolution,
but keeps reusable logic inside `ia_analysis.orbits`.

Pipeline:

1. Build a synthetic group-subhalo orbit template.
2. Attach an ellipsoidal group model and analytic tidal tensor proxy.
3. Compute instantaneous and delayed tidal stripping histories.
4. Draw orbit, radial motion, mass stripping, and shape-evolution figures.
5. Save figures into `outputs/orbit_template_demo`.

The math notation uses inline dollar delimiters, for example $r_t$, $M/M_0$,
$V_\mathrm{max}$, and $r_\mathrm{max}$.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start=None):
    """Find the repository root from a notebook working directory."""
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "src" / "ia_analysis").exists():
            return path
    return start


PROJECT_ROOT = find_project_root()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "orbit_template_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, OUTPUT_DIR

In [ ]:
import numpy as np
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs matplotlib to render and save figures. "
        "Install the project plotting dependencies before running it."
    ) from exc

from ia_analysis.orbits import (
    EllipsoidalGroupModel,
    TreeTrack,
    build_orbit_template,
    homogeneous_ellipsoid_tidal_tensor,
    initial_shape_alignment_model,
)
from ia_analysis.orbits.tidal_stripping import (
    TidalStrippingOptions,
    stripping_history_from_template,
    stripping_summary,
    template_host_curvature_powerlaw,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Synthetic Orbit Template

The synthetic track is intentionally small and deterministic.  It is not a
calibrated cosmological orbit; it is a stable visual test for the template,
shape, and stripping interfaces.

In [ ]:
rng = np.random.default_rng(14)
n_sample = 240
time = np.linspace(0.0, 6.0, n_sample)
phase = 3.4 * np.pi * time / time[-1]
radius = 1.08 - 0.42 * np.cos(phase) + 0.08 * np.cos(2.0 * phase)

relative_position = np.column_stack([
    radius * np.cos(phase),
    0.72 * radius * np.sin(phase),
    0.12 * np.sin(1.7 * phase),
])
relative_velocity = np.gradient(relative_position, time, axis=0)

host_position = np.zeros_like(relative_position)
host_velocity = np.zeros_like(relative_velocity)
host_mass = 20.0 * (1.0 + 0.04 * time / time[-1])
subhalo_mass = np.ones_like(time)

group_track = TreeTrack(
    object_id="demo_group",
    snapshots=time,
    positions=host_position,
    velocities=host_velocity,
    mass=host_mass,
)
subhalo_track = TreeTrack(
    object_id="demo_subhalo",
    snapshots=time,
    positions=relative_position,
    velocities=relative_velocity,
    mass=subhalo_mass,
)

template = build_orbit_template(group_track, subhalo_track)
r = template.radius
speed = template.speed

print(f"Template samples: {r.size}")
print(f"Radius range: {r.min():.3f} to {r.max():.3f}")
print(f"Speed range: {speed.min():.3f} to {speed.max():.3f}")

## Ellipsoidal Group And Shape Proxy

The group is represented by a homogeneous ellipsoid.  The notebook starts with
coherent inner and outer shapes aligned with the group/tidal tensor, then adds a
simple stripping-dependent flattening trend for visualization.

In [ ]:
def rotation_z(angle_rad):
    """Return a right-handed rotation matrix around the z-axis."""
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])


group_model = EllipsoidalGroupModel(
    axes=(1.55, 1.02, 0.68),
    orientation=rotation_z(np.deg2rad(24.0)),
    mass=host_mass[-1],
)

tidal_tensor = homogeneous_ellipsoid_tidal_tensor(group_model)
shape_layers = initial_shape_alignment_model(group_model, mode="tidal_aligned")

evals = np.linalg.eigvalsh(tidal_tensor)
print("Ellipsoidal tidal eigenvalues:", np.round(evals, 4))
print("Available shape layers:", sorted(shape_layers))

## Tidal Stripping Histories

The instantaneous curve is the Jacobi/power-law target.  The delayed curve is
the recommended lightweight model because it strips toward that target over a
finite orbital response time.  Both curves retain a small remnant floor and keep
$M/M_0$ monotonic.

In [ ]:
host_curvature = template_host_curvature_powerlaw(
    r,
    amplitude=0.95,
    scale_radius=0.32,
    exponent=3.0,
)

instant_options = TidalStrippingOptions(
    mode="instantaneous_powerlaw",
    density_slope=2.0,
    minimum_bound_fraction=1.0e-3,
)
delayed_options = TidalStrippingOptions(
    mode="delayed_powerlaw",
    density_slope=2.0,
    tau_orbits=0.35,
    minimum_bound_fraction=1.0e-3,
)

instant_history = stripping_history_from_template(
    template,
    time=time,
    mass0=1.0,
    reference_radius=0.54,
    host_curvature=host_curvature,
    options=instant_options,
    gravitational_constant=1.0,
)
delayed_history = stripping_history_from_template(
    template,
    time=time,
    mass0=1.0,
    reference_radius=0.54,
    host_curvature=host_curvature,
    options=delayed_options,
    gravitational_constant=1.0,
)

stripping_summary(delayed_history)

In [ ]:
mass_loss = 1.0 - delayed_history.mass_fraction
inner_axis_ratio = 0.88 - 0.08 * mass_loss
outer_axis_ratio = 0.74 - 0.16 * mass_loss
major_axis_angle = np.rad2deg(
    np.unwrap(np.arctan2(relative_position[:, 1], relative_position[:, 0]))
)
major_axis_angle = major_axis_angle - major_axis_angle[0]
major_axis_angle = 0.18 * major_axis_angle + 4.0 * np.sin(phase)

shape_proxy = {
    "inner_axis_ratio": inner_axis_ratio,
    "outer_axis_ratio": outer_axis_ratio,
    "major_axis_angle_deg": major_axis_angle,
}

## Figures

The following cells save demonstration figures.  Re-run the notebook after code
changes and inspect these images before using the orbit-template workflow in a
larger analysis.

In [ ]:
def savefig(fig, name):
    """Save a figure to the demo output directory and return its path."""
    path = OUTPUT_DIR / name
    fig.tight_layout()
    fig.savefig(path)
    return path


fig, ax = plt.subplots(figsize=(6.0, 5.4))
angles = np.linspace(0.0, 2.0 * np.pi, 240)
ellipse = np.column_stack([
    group_model.axes[0] * np.cos(angles),
    group_model.axes[1] * np.sin(angles),
    np.zeros_like(angles),
]) @ group_model.orientation.T

scatter = ax.scatter(
    relative_position[:, 0],
    relative_position[:, 1],
    c=time,
    s=11,
    cmap="viridis",
    linewidths=0,
    label="orbit samples",
)
ax.plot(ellipse[:, 0], ellipse[:, 1], color="black", lw=1.4, label="group ellipsoid")
ax.scatter(relative_position[0, 0], relative_position[0, 1], marker="o", s=52, color="#2a9d8f", label="start")
ax.scatter(relative_position[-1, 0], relative_position[-1, 1], marker="s", s=52, color="#e76f51", label="end")
ax.scatter(relative_position[np.argmin(r), 0], relative_position[np.argmin(r), 1], marker="*", s=100, color="#f4a261", label="pericentre")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x / template length")
ax.set_ylabel("y / template length")
ax.set_title("Group-centric orbit in an ellipsoidal host")
fig.colorbar(scatter, ax=ax, label="time")
ax.legend(loc="upper right", fontsize=8)
orbit_path = savefig(fig, "orbit_template_path.png")
orbit_path

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7.0, 6.0), sharex=True)
axes[0].plot(time, r, color="#264653", lw=1.8)
axes[0].set_ylabel("radius")
axes[0].set_title("Template orbital motion")
axes[1].plot(time, speed, color="#2a9d8f", lw=1.8)
axes[1].set_ylabel("speed")
axes[1].set_xlabel("time")
radius_path = savefig(fig, "orbit_radius_speed.png")
radius_path

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7.0, 6.2), sharex=True)
axes[0].plot(time, instant_history.mass_fraction, color="#d62828", lw=1.5, label="instantaneous target")
axes[0].plot(time, delayed_history.mass_fraction, color="#003049", lw=2.0, label="delayed bound mass")
axes[0].set_ylabel("$M/M_0$")
axes[0].set_title("Tidal stripping history")
axes[0].legend(fontsize=8)
axes[1].plot(time, delayed_history.tidal_radius, color="#f77f00", lw=1.8, label="$r_t$")
axes[1].plot(time, r, color="#6c757d", lw=1.2, alpha=0.8, label="orbit radius")
axes[1].set_ylabel("radius")
axes[1].set_xlabel("time")
axes[1].legend(fontsize=8)
stripping_path = savefig(fig, "tidal_stripping_history.png")
stripping_path

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7.0, 6.2), sharex=True)
axes[0].plot(time, shape_proxy["inner_axis_ratio"], color="#457b9d", lw=1.8, label="inner")
axes[0].plot(time, shape_proxy["outer_axis_ratio"], color="#a23e48", lw=1.8, label="outer")
axes[0].set_ylabel("axis ratio")
axes[0].set_title("Shape proxy tied to stripping")
axes[0].legend(fontsize=8)
axes[1].plot(time, shape_proxy["major_axis_angle_deg"], color="#2b2d42", lw=1.8)
axes[1].set_ylabel("major-axis angle / deg")
axes[1].set_xlabel("time")
shape_path = savefig(fig, "shape_evolution_proxy.png")
shape_path

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.0, 7.2))
axes = axes.ravel()
axes[0].scatter(relative_position[:, 0], relative_position[:, 1], c=time, s=7, cmap="viridis")
axes[0].plot(ellipse[:, 0], ellipse[:, 1], color="black", lw=1.1)
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_title("Orbit")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")

axes[1].plot(time, r, color="#264653", label="radius")
axes[1].plot(time, speed / np.nanmax(speed) * np.nanmax(r), color="#2a9d8f", label="scaled speed")
axes[1].set_title("Motion")
axes[1].set_xlabel("time")
axes[1].legend(fontsize=8)

axes[2].plot(time, instant_history.mass_fraction, color="#d62828", lw=1.3, label="instant")
axes[2].plot(time, delayed_history.mass_fraction, color="#003049", lw=1.8, label="delayed")
axes[2].plot(time, delayed_history.vmax_ratio, color="#f77f00", lw=1.3, label="$V_\mathrm{max}$ ratio")
axes[2].set_ylim(0.0, 1.05)
axes[2].set_title("Stripping")
axes[2].set_xlabel("time")
axes[2].legend(fontsize=8)

axes[3].plot(time, inner_axis_ratio, color="#457b9d", label="inner q")
axes[3].plot(time, outer_axis_ratio, color="#a23e48", label="outer q")
axes[3].set_title("Shape layers")
axes[3].set_xlabel("time")
axes[3].legend(fontsize=8)

summary_path = savefig(fig, "orbit_shape_stripping_summary.png")
summary_path

## Interpretation And Next Steps

The delayed model preserves the old notebook's intuitive $r_t$ diagnostic while
avoiding an all-at-once mass drop at pericentre.  For production work, replace
the demo curvature with an NFW or ellipsoidal host Hessian, then calibrate the
stripping timescale and tidal-track coefficients against controlled simulations
or DASH-like template libraries.